# BridgeIDS walkthrough
Legacy NSL-KDD → bridge ForgeIIoT → modern Edge-IIoTset in 8 short steps. Run after `setup_*` (or `pip install -r requirements.txt && pip install -e .`).
Full experiments: `python -m bridgeids.run exp all`.

In [ ]:
import sys, pathlib
root = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()
sys.path.insert(0, str(root / 'src'))
import numpy as np, pandas as pd
from bridgeids.data.loaders import load
from bridgeids.features.views import canonical_view, cbs_from_view, pbs_from_view
nsl, forge, edge = load('nsl'), load('forge'), load('edge')
print(nsl.shape, forge.shape, edge.shape)

## 1 · One semantic view for connections *and* packets

In [ ]:
for name, df in [('nsl', nsl), ('forge', forge), ('edge', edge)]:
    v = canonical_view(df.sample(5, random_state=1), name)
    print(name); print(v[['proto','service','syn','ack','rst','syn_only','bytes','app_mqtt']].to_string(), '\n')

## 2 · The language interlingua

In [ ]:
from bridgeids.nlp.flow2text import records_to_text
for name, df in [('nsl', nsl), ('forge', forge), ('edge', edge)]:
    print(name, '→', records_to_text(df.sample(3, random_state=2), name))

## 3 · Cross-dataset test in the Canonical Behavioural Space

In [ ]:
from bridgeids.experiments.common import get_split
from bridgeids.models.zoo import make_model, proba_pos
from sklearn.metrics import roc_auc_score
sp = {n: get_split(n, 'cbs', 42, cap_train=20000, cap_test=10000) for n in ['nsl','forge','edge']}
for src in sp:
    m = make_model('lgbm').fit(sp[src].Xtr.values, sp[src].ytr)
    print(src, {tgt: round(roc_auc_score(sp[tgt].yte, proba_pos(m, sp[tgt].Xte.values)), 3) for tgt in sp})

## 4 · Drift between datasets (proxy-A-distance)

In [ ]:
from bridgeids.drift import domain_classifier_auc
for a, b in [('nsl','edge'), ('forge','edge')]:
    auc, pad = domain_classifier_auc(sp[a].Xtr.values, sp[b].Xtr.values)
    print(f'{a} vs {b}: domain-classifier AUC={auc:.3f}  proxy-A-distance={pad:.3f}')

## 5 · Hybrid zero-day detector (supervised + benign-calibrated anomaly)

In [ ]:
from bridgeids.models.hybrid import HybridDetector
e = get_split('edge', 'pbs', 42, cap_train=20000, cap_test=10000)
keep = e.mtr['label_fine'].values != 'XSS'                 # XSS never seen in training
h = HybridDetector().fit(e.Xtr.values[keep], e.ytr[keep])
s = h.score(e.Xte.values)
xss = e.mte['label_fine'].values == 'XSS'
print('XSS detection rate @0.5:', (s[xss] >= 0.5).mean().round(3), ' benign FPR:', (s[e.yte == 0] >= 0.5).mean().round(4))

## 6 · SHAP explanation as a narrative

In [ ]:
from bridgeids.xai.explain import shap_values_binary, narrative
i = int(np.where(xss)[0][0])
sv = shap_values_binary(h.sup, e.Xte.values[i:i+1])[0]
top = np.argsort(-np.abs(sv))[:4]
print(narrative(float(s[i]), [(e.Xte.columns[k], float(e.Xte.values[i, k]), float(sv[k])) for k in top], record_id=i))

## 7 · Dynamic detection in-process (no network needed)

In [ ]:
from bridgeids.utils import load_config
from bridgeids.stream.artifacts import load_artifacts
from bridgeids.stream.engine import DetectionEngine
from bridgeids.stream.replayer import build_stream
cfg = load_config()
eng = DetectionEngine(load_artifacts(cfg=cfg), cfg, explain=True)
stream = build_stream('edge', 'campaign', n=4000, seed=0)
for k in range(0, len(stream), 256):
    eng.process(stream[k:k+256])
snap = eng.snapshot()
print({k: snap[k] for k in ['n','alerts','f1','mcc','fpr','latency_ms_p95']})
print(snap['family_counts'])
print(snap['recent_alerts'][0]['explanation'])

## 8 · Live system
```bash
python -m bridgeids.run serve            # http://127.0.0.1:8000
python -m bridgeids.run replay --scenario campaign --rate 400
```